# Notebook 2 – Snapshot Creation
Conclusion

• Dataset successfully cleaned and validated.

• Missing and invalid records were handled appropriately.

• Clean dataset prepared for the next stage of analysis.

In [ ]:
!pwd

/content


In [ ]:
!ls

sample_data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

retail = pd.read_pickle(
    "/content/drive/MyDrive/Project files/retail_clean.pkl"
)

print(retail.shape)
retail.head()

(805549, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,purchase_month
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12


In [ ]:
#Block 1 — Define Snapshot Date Range

import pandas as pd

# Find the earliest transaction date in the dataset
dataset_start = retail["InvoiceDate"].min()

# Find the latest transaction date in the dataset
dataset_end = retail["InvoiceDate"].max()

# We need 180 days of history before a snapshot
# Therefore the first possible snapshot is:
first_snapshot_date = dataset_start + pd.Timedelta(days=180)

# We need 90 days of future data after a snapshot
# Therefore the last possible snapshot is:
last_snapshot_date = dataset_end - pd.Timedelta(days=90)

# Print results to verify
print("Dataset Start:", dataset_start)
print("Dataset End:", dataset_end)

print("First Valid Snapshot:", first_snapshot_date)
print("Last Valid Snapshot:", last_snapshot_date)

Dataset Start: 2009-12-01 07:45:00
Dataset End: 2011-12-09 12:50:00
First Valid Snapshot: 2010-05-30 07:45:00
Last Valid Snapshot: 2011-09-10 12:50:00


In [ ]:
#Block 2 — Create Monthly Snapshot Dates
# Create month-end dates between first and last valid snapshot

snapshot_calendar = pd.date_range(
    start=first_snapshot_date,
    end=last_snapshot_date,
    freq="M"  # M = month end
)

# How many snapshot dates do we have?
print("Number of Snapshot Dates:")
print(len(snapshot_calendar))

# Display all snapshot dates
print(snapshot_calendar)

Number of Snapshot Dates:
16
DatetimeIndex(['2010-05-31 07:45:00', '2010-06-30 07:45:00',
               '2010-07-31 07:45:00', '2010-08-31 07:45:00',
               '2010-09-30 07:45:00', '2010-10-31 07:45:00',
               '2010-11-30 07:45:00', '2010-12-31 07:45:00',
               '2011-01-31 07:45:00', '2011-02-28 07:45:00',
               '2011-03-31 07:45:00', '2011-04-30 07:45:00',
               '2011-05-31 07:45:00', '2011-06-30 07:45:00',
               '2011-07-31 07:45:00', '2011-08-31 07:45:00'],
              dtype='datetime64[ns]', freq='ME')


/tmp/ipykernel_7234/476370344.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  snapshot_calendar = pd.date_range(


In [ ]:
#Block 3 — Find Each Customer's First Purchase
# For every customer, find the date of their first purchase

customer_first_purchase = (
    retail.groupby("Customer ID")["InvoiceDate"]
    .min()
    .reset_index()
)

# Rename columns for readability

customer_first_purchase.columns = [
    "Customer ID",
    "first_purchase_date"
]

# Check first few rows
customer_first_purchase.head()

,Customer ID,first_purchase_date
0,12346,2009-12-14 08:34:00
1,12347,2010-10-31 14:20:00
2,12348,2010-09-27 14:59:00
3,12349,2010-04-29 13:20:00
4,12350,2011-02-02 16:01:00


In [ ]:
#Block 4 — Generate Valid Customer Snapshots
# Empty list to store snapshot tables
snapshot_rows = []

# Loop through every snapshot date
for snapshot_date in snapshot_calendar:

    # Keep only customers who have at least
    # 180 days of history before this snapshot

    eligible_customers = customer_first_purchase[
        (
            snapshot_date
            - customer_first_purchase["first_purchase_date"]
        ).dt.days >= 180
    ]

    # Create a copy so we don't modify original data
    temp = eligible_customers.copy()

    # Add the current snapshot date
    temp["snapshot_date"] = snapshot_date

    # Keep only required columns
    temp = temp[
        ["Customer ID", "snapshot_date"]
    ]

    # Store results
    snapshot_rows.append(temp)

In [ ]:
#Block 5 — Combine All Snapshots
# Combine all monthly snapshot tables
# into one large dataset

snapshot_df = pd.concat(
    snapshot_rows,
    ignore_index=True
)

# Display first few rows
snapshot_df.head()

,Customer ID,snapshot_date
0,12362,2010-05-31 07:45:00
1,12490,2010-05-31 07:45:00
2,12533,2010-05-31 07:45:00
3,12636,2010-05-31 07:45:00
4,12682,2010-05-31 07:45:00


In [ ]:
#Block 6 — Audit the Snapshot Dataset
# Total number of snapshot rows
print("Snapshot Rows:")
print(len(snapshot_df))

# Number of unique customers
print("Unique Customers:")
print(snapshot_df["Customer ID"].nunique())

# Number of monthly snapshot dates
print("Unique Snapshot Dates:")
print(snapshot_df["snapshot_date"].nunique())

# Earliest snapshot date
print("First Snapshot:")
print(snapshot_df["snapshot_date"].min())

# Latest snapshot date
print("Last Snapshot:")
print(snapshot_df["snapshot_date"].max())

Snapshot Rows:
46098
Unique Customers:
4557
Unique Snapshot Dates:
16
First Snapshot:
2010-05-31 07:45:00
Last Snapshot:
2011-08-31 07:45:00


In [ ]:
snapshot_df["snapshot_date"] = (
    pd.to_datetime(
        snapshot_df["snapshot_date"]
    ).dt.normalize()
)

In [ ]:
# ====================================================
# SAVE SNAPSHOT DATASET
# ====================================================

snapshot_df.to_pickle(
    "/content/drive/MyDrive/Project files/snapshot_df.pkl"
)

print("snapshot_df saved successfully")

snapshot_df saved successfully
